In [ ]:
import pandas as pd
import numpy as np
import torch
import torch.nn.functional as F
import torch.nn as nn
from tqdm import tqdm
import matplotlib.pyplot as plt
from transformers import AutoTokenizer

In [ ]:
df_train = pd.read_csv("train_processed.csv")
df_test = pd.read_csv("test_processed.csv")

In [ ]:
MAX_CHARS = 10000
MODEL_TOKENIZER = "bert-base-uncased"
SEED = 42
NUM_CLASSES = 2
BATCH_SIZE = 32
EPOCHS = 100
LR = 0.0002
MAX_LENGTH = 1024
DROPOUT = 0.2
EMBEDDING_DIM = 100

In [ ]:
# Convert columns to string and integer types to prevent errors
df_train['file_1'] = df_train['file_1'].astype(str)
df_train['file_2'] = df_train['file_2'].astype(str)
df_train['real_text_id'] = df_train['real_text_id'].astype(int)

In [ ]:
texts = []
labels = []

for _, row in df_train.iterrows():
    if row['real_text_id'] == 1:
        texts.append(row['file_1'][:MAX_CHARS])
        labels.append(1)
        texts.append(row['file_2'][:MAX_CHARS])
        labels.append(0)
    else:
        texts.append(row['file_1'][:MAX_CHARS])
        labels.append(0)
        texts.append(row['file_2'][:MAX_CHARS])
        labels.append(1)


data = pd.DataFrame({'text': texts,
                     'label': labels})

In [ ]:
data.head()

,text,label
0,virsa visible infrare survey telescope array p...,1
1,china relay network release significant petaby...,0
2,china goal project involve achieve accuracy le...,0
3,project aim achieve accuracy level dex analyze...,1
4,scientist learn galaxy form evolve method obse...,1


In [ ]:
# Tokenizer
tokenizer = AutoTokenizer.from_pretrained(MODEL_TOKENIZER,
                                          trust_remote_code=True)

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [ ]:
def function_tokenize(example):
    return tokenizer(
        example["text"],
        max_length=MAX_LENGTH,
        truncation=True,
        padding='max_length')

In [ ]:
train_data = data.sample(frac=0.8, random_state=SEED).reset_index(drop=True)
test_data = data.drop(train_data.index).reset_index(drop=True)

labels_train = train_data["label"]
labels_test = test_data["label"]

train_data = train_data.apply(function_tokenize, axis=1)
test_data = test_data.apply(function_tokenize, axis=1)

train_input_ids = torch.tensor([train_data[i]["input_ids"] for i in train_data.index])
test_input_ids = torch.tensor([test_data[i]["input_ids"] for i in test_data.index])

train_data["input_ids"] = train_input_ids
test_data["input_ids"] = test_input_ids

train_data["label"] = labels_train
test_data["label"] = labels_test

In [ ]:
class ModelClassificator(nn.Module):
    def __init__(self, num_classes, embedding, embedding_dim, dropout):
        super().__init__()

        self.embedding = embedding
        self.dropout = dropout
        self.gap = nn.AdaptiveAvgPool1d(1)

        self.fc1 = nn.Linear(embedding_dim, 128)
        self.fc2 = nn.Linear(128, 256)
        self.fc3 = nn.Linear(256, 512)
        self.fc4 = nn.Linear(512, 768)
        self.fc5 = nn.Linear(768, num_classes)

    def forward(self, x):
        x = self.embedding(x)
        x = x.transpose(1, 2)

        x = self.gap(x)
        x = x.view(x.size(0), -1)

        x = F.relu(self.fc1(x))
        x = F.dropout(x, p=self.dropout, training=self.training)

        x = F.relu(self.fc2(x))
        x = F.dropout(x, p=self.dropout, training=self.training)

        x = F.relu(self.fc3(x))
        x = F.dropout(x, p=self.dropout, training=self.training)

        x = F.relu(self.fc4(x))
        x = F.dropout(x, p=self.dropout, training=self.training)

        return self.fc5(x)

In [ ]:
GLOVE_FILE = 'glove.6B.100d.txt'

GLOVE_EMBEDDINGS = {}

with open(GLOVE_FILE) as f:
    for line in f:
        values = line.split()
        word = values[0]
        coefs = np.asarray(values[1:], dtype='float32')
        GLOVE_EMBEDDINGS[word] = coefs

VOCAB_SIZE = tokenizer.vocab_size
word_index = tokenizer.vocab

EMBEDDINGS_MATRIX = np.zeros((VOCAB_SIZE+1, EMBEDDING_DIM))

for word, i in word_index.items():
    embedding_vector = GLOVE_EMBEDDINGS.get(word)
    if embedding_vector is not None:
        EMBEDDINGS_MATRIX[i] = embedding_vector

embedding = nn.Embedding.from_pretrained(
    torch.tensor(EMBEDDINGS_MATRIX, dtype=torch.float32),
    freeze=False,
    padding_idx=0
)

In [ ]:
model = ModelClassificator(NUM_CLASSES, embedding, EMBEDDING_DIM, DROPOUT)

In [ ]:
class TextDataset(torch.utils.data.Dataset):
    def __init__(self, df):
        self.texts = df["input_ids"]
        self.labels = df["label"]

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        return torch.tensor(self.texts[idx], dtype=torch.long), torch.tensor(self.labels[idx], dtype=torch.long)

In [ ]:
train_dataset = TextDataset(train_data)
test_dataset = TextDataset(test_data)

train_loader = torch.utils.data.DataLoader(train_dataset,
                                           batch_size=BATCH_SIZE,
                                           shuffle=True)

test_loader = torch.utils.data.DataLoader(test_dataset,
                                          batch_size=BATCH_SIZE)

In [1]:
class Trainer:
    def __init__(self,
                 model, num_epochs, optimizer, criterion,
                 train_data, test_data):

        self.model = model
        self.num_epochs = num_epochs
        self.optimizer = optimizer
        self.criterion = criterion
        self.scheduler = scheduler
        self.train_data = train_data
        self.test_data = test_data
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        self.train_losses, self.test_losses = [], []
        self.train_accuracies, self.test_accuracies = [], []

    def train_epoch(self, epoch):
        """
        Executa o treinamento do modelo por uma época.

        Args:
            epoch: Número da época atual

        Returns:
            tuple: Média da perda e acurácia de treinamento
        """
        self.model.train()
        running_loss = 0.0
        correct = 0
        total = 0
        pbar = tqdm(self.train_data, desc=f'Epoch {epoch+1}/{self.num_epochs} [Train]')
        for data, target in pbar:
            data, target = data.to(self.device), target.to(self.device)
            self.optimizer.zero_grad() # Zera os gradientes acumulados
            output = self.model(data) # Passa os dados pelo modelo para obter as previsões
            loss = self.criterion(output, target) # Calcula a perda entre as previsões e os valores reais
            loss.backward()  # Calcula os gradientes da perda em relação aos parâmetros
            self.optimizer.step()  # Atualiza os pesos com base nos gradientes calculados

            running_loss += loss.item() # Acumula a perda para calcular a média
            _, predicted = output.max(1) # Encontra a classe com a maior probabilidade
            total += target.size(0) # Atualiza o total de amostras
            correct += predicted.eq(target).sum().item() # Conta o número de previsões corretas
            pbar.set_postfix({'Loss': f'{loss.item():.4f}', 'Acc': f'{100.*correct/total:.2f}%'}) # Atualiza o progresso

        return running_loss / len(self.train_data), 100. * correct / total # Retorna a média da perda e a acurácia

    def test_epoch(self):
        """
        Executa a avaliação do modelo.

        Returns:
            tuple: Média da perda e acurácia de teste
        """
        self.model.eval()  # Coloca o modelo em modo de avaliação (sem dropout, BN fixo)
        test_loss = 0
        correct = 0
        total = 0
        with torch.no_grad():
            pbar = tqdm(self.test_data, desc='Testing')
            for data, target in pbar:
                data, target = data.to(self.device), target.to(self.device) # Move os dados para o dispositivo
                output = self.model(data) # Passa os dados pelo modelo para obter as previsões
                test_loss += self.criterion(output, target).item() # Calcula a perda entre as previsões e os valores reais
                _, predicted = output.max(1) # Encontra a classe com a maior probabilidade
                total += target.size(0)
                correct += predicted.eq(target).sum().item()
                pbar.set_postfix({'Loss': f'{test_loss/len(self.test_data):.4f}', 'Acc': f'{100.*correct/total:.2f}%'})

        return test_loss / len(self.test_data), 100. * correct / total

    def train_test(self):
        """
        Executa o treinamento e avaliação do modelo.

        Returns:
            dict: Dicionário contendo métricas do treinamento e avaliação
        """
        self.model.to(self.device) # Move o modelo para o dispositivo

        model_name = self.model.__class__.__name__ # Obtém o nome do modelo
        print(f"Iniciando treinamento do modelo {model_name}")

        best_accuracy = 0.0

        for epoch in range(self.num_epochs):
            train_loss, train_acc = self.train_epoch(epoch) # Treina o modelo por uma época
            test_loss, test_acc = self.test_epoch() # Avalia o modelo no conjunto de teste

            self.scheduler.step() # Atualiza a taxa de aprendizado

            self.train_losses.append(train_loss) # Armazena a perda de treinamento
            self.train_accuracies.append(train_acc) # Armazena a acurácia de treinamento
            self.test_losses.append(test_loss) # Armazena a perda de teste
            self.test_accuracies.append(test_acc) # Armazena a acurácia de teste

            if test_acc > best_accuracy:
                best_accuracy = test_acc

            print(f"Epoch {epoch+1}: Train Loss: {train_loss:.4f}, Acc: {train_acc:.2f}% | "
                  f"Test Loss: {test_loss:.4f}, Acc: {test_acc:.2f}% | "
                  f"LR: {self.optimizer.param_groups[0]['lr']:.6f}")

        print(f"\nMelhor acurácia de teste: {best_accuracy:.2f}%")
        torch.save(self.model.state_dict(), f'{model_name.lower()}.pth')
        print(f"\nModelo salvo como {model_name.lower()}.pth")

        self.metrics = {
            'best_accuracy': best_accuracy,
            'train_losses': self.train_losses,
            'test_losses': self.test_losses,
            'train_accuracies': self.train_accuracies,
            'test_accuracies': self.test_accuracies
        }

    def __call__(self):
       return self.train_test()

    def plot_results(self):
        _, (ax1, ax2) = plt.subplots(1, 2, figsize=(18, 6)) # Cria uma figura com dois subplots
        ax1.plot(self.train_losses, label='Train Loss') # Plota a perda de treinamento
        ax1.plot(self.test_losses, label='Test Loss') # Plota a perda de teste
        ax1.set_title('Histórico de Loss') # Define o título do primeiro subplot
        ax1.set_xlabel('Época') # Define o rótulo do eixo x do primeiro subplot
        ax1.set_ylabel('Loss') # Define o rótulo do eixo y do primeiro subplot
        ax1.legend() # Adiciona uma legenda ao primeiro subplot
        ax1.grid(True) # Adiciona uma grade ao primeiro subplot

        ax2.plot(self.train_accuracies, label='Train Accuracy') # Plota a acurácia de treinamento
        ax2.plot(self.test_accuracies, label='Test Accuracy') # Plota a acurácia de teste
        ax2.set_title('Histórico de Acurácia') # Define o título do segundo subplot
        ax2.set_xlabel('Época') # Define o rótulo do eixo x do segundo subplot
        ax2.set_ylabel('Acurácia (%)') # Define o rótulo do eixo y do segundo subplot
        ax2.legend() # Adiciona uma legenda ao segundo subplot
        ax2.grid(True)

        plt.tight_layout()
        plt.savefig('training_results_improved.png', dpi=300)
        plt.show()

In [ ]:
optimizer=torch.optim.AdamW(model.parameters(), lr=LR)
criterion=nn.CrossEntropyLoss()
scheduler=torch.optim.lr_scheduler.CosineAnnealingLR(optimizer,
                                                    T_max=EPOCHS)

In [ ]:
trainer = Trainer(
    model=model,
    num_epochs=EPOCHS,
    optimizer=optimizer,
    criterion=criterion,
    scheduler=scheduler,
    train_data=train_loader,
    test_data=test_loader,
)

In [ ]:
trainer()

In [ ]:
trainer.plot_results()

In [ ]:
path_best_model = "/content/best_modelclassificator.pth"

model = ModelClassificator(num_classes, embedding)
model.load_state_dict(torch.load(path_best_model))
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
model.eval()

ModelClassificator(
  (gap): AdaptiveAvgPool1d(output_size=1)
  (embedding): Embedding(30523, 768)
  (linear1): Linear(in_features=768, out_features=512, bias=True)
  (linear2): Linear(in_features=512, out_features=256, bias=True)
  (classifier): Linear(in_features=256, out_features=2, bias=True)
  (relu): ReLU()
  (dropout): Dropout(p=0.3, inplace=False)
)

In [ ]:
def inference_pipeline(text):
    inputs = tokenizer(text, truncation=True,
                       max_length=max_length,
                       padding="max_length",
                       return_tensors="pt")

    with torch.no_grad():
        inputs = inputs["input_ids"].to(device)
        probs = model(inputs)
        probs = probs.detach().to('cpu')
        probs = F.softmax(probs, dim=1)
        return probs

def get_real_idx(prob_fake_file1, prob_fake_file2):
    if prob_fake_file1 < prob_fake_file2:
        return 1
    else:
        return 2

In [ ]:
df_test["logits1"] = df_test.apply(lambda x: float(inference_pipeline(x["file_1"][:MAX_CHARS])[0][0]), axis=1)
df_test["logits2"] = df_test.apply(lambda x: float(inference_pipeline(x["file_2"][:MAX_CHARS])[0][0]), axis=1)
df_test['real_text_id'] = df_test.apply(lambda x: get_real_idx(x['logits1'], x['logits2']), axis=1)

In [ ]:
df_sub = df_test[['id', 'real_text_id']]
df_sub.to_csv("submission_05_07_sb3.csv", index=False)